# F2b · gate GT: Qwen3.8-27B pensando contra os decisores (Kaggle, T4 x2)

**A pergunta** (issue #75, ADR 0004): o Qwen3.8-27B em modo thinking acerta pelo menos **+10 pontos** a mais que o melhor decisor de um passe, com o piso do IC 95% (bootstrap pareado) **acima de 0**, nas 400 linhas da amostra pública (10 provas de conhecimento e raciocínio, 40 por prova)?

Este notebook roda o pensador em 4-bit (NF4, cálculo em fp16) com `transformers`, uma linha por vez e com retomada, e termina na linha `GT VERDICT`. O guia (tempos, limites, o que fazer com cada resultado) está em `decide/think/README.md`.

**Antes de rodar**
1. Settings > Accelerator: **GPU T4 x2**. Settings > Internet: **On**.
2. Add Input: um dataset **privado** com o `pack.jsonl.gz`. Ele tem o texto das provas (GPQA e BBH pedem para não publicar): nunca torne público o dataset, nem este notebook com as saídas.
3. O tempo **não foi medido** numa T4 (UNVERIFIED). A primeira execução faz só 3 linhas (`LIMIT = "3"`): leia os tokens/s e o ETA que ela imprime e só então ponha `LIMIT = ""`.
4. Se a saída vier como ruído, o relatório avisa. Um FAIL feito de ruído não é veredito (README, "Se a saída for ruído").
5. Tudo que este notebook gera é dado de avaliação: **nunca entra em treino** (ADR 0004, item 6).

In [ ]:
# 1. Parâmetros: a única célula que você edita.
SIMPLICIO_SHA = "b6778808fb5cfb551d60ac6081274a061af20d74"   # commit de simpletibr/simplicio-27b com decide/think (fixado)
PACK = "/kaggle/input/think-gt-pack/pack.jsonl.gz"   # dataset privado; se o caminho estiver errado, a célula 4 procura em /kaggle/input
WORK = "/kaggle/working/think-gt"                    # results.jsonl e report/ ficam aqui
HF_CACHE = "/kaggle/temp/hf"      # o checkpoint bf16 tem 51,7 GiB e /kaggle/working é limitado a 20 GB (UNVERIFIED qual caminho tem espaço: a célula 4 imprime)
MAX_TOKENS = 16384                # orçamento de pensamento por pergunta; linha que bate nele conta como erro e é listada
BATCH_SIZE = 4                    # linhas geradas juntas; o lote dura o tempo da resposta mais longa (se faltar memória, o lote é dividido sozinho)
TIME_BUDGET_H = 8.0               # para de começar linhas depois disso; fique abaixo do limite da sua sessão
LIMIT = "3"                       # 1a execução: 3 linhas para medir o tempo. Depois: "" (todas as pendentes)
RESUME_FROM = "/kaggle/input/think-gt-results/results.jsonl"   # resultados de uma sessão anterior (opcional)

TIME_BUDGET_S = int(TIME_BUDGET_H * 3600)
EXTRA = f"--limit {LIMIT}" if LIMIT else ""

## 2. Código fixado
O código vem do commit `SIMPLICIO_SHA`, nunca de `main`. Depois que o PR for mergeado, troque o SHA pelo do commit de merge em `main`.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/simplicio-27b"          # REPO_PLACEHOLDER
os.makedirs(REPO, exist_ok=True)
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "init", "-q", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "https://github.com/simpletibr/simplicio-27b.git", SIMPLICIO_SHA], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == SIMPLICIO_SHA, f"checked out {head}, expected {SIMPLICIO_SHA}"
os.chdir(REPO)
print("code at", head)

## 3. Dependências (versões exatas)
São as mesmas do Colab do repositório (issue #31). Que elas se resolvam com o `torch` desta imagem do Kaggle é **UNVERIFIED**; o log abaixo mostra o que foi instalado.

In [ ]:
!pip install -q "transformers==5.5.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "huggingface_hub==1.33.0"
!pip list --format=freeze | grep -i -E "^(torch|transformers|accelerate|bitsandbytes|huggingface-hub)=="

## 4. Conferências antes de gastar GPU

In [ ]:
import glob, shutil

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().splitlines()
print(gpus)
total_mib = sum(int(g.split(",")[1].split()[0]) for g in gpus) if gpus else 0
assert total_mib >= 28 * 1024, "preciso de uns 30 GiB de GPU no total: escolha Accelerator = GPU T4 x2"

if not os.path.exists(PACK):
    found = glob.glob("/kaggle/input/**/pack.jsonl.gz", recursive=True)
    assert found, "pack.jsonl.gz não encontrado: use Add Input com o seu dataset privado"
    PACK = found[0]
print("pack:", PACK, f"{os.path.getsize(PACK) / 1e3:.0f} KB")

os.makedirs(WORK, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
free_gib = shutil.disk_usage(HF_CACHE).free / 2**30
print(f"disco livre para o cache do modelo ({HF_CACHE}): {free_gib:.0f} GiB; o checkpoint bf16 tem 51,7 GiB")
assert free_gib > 58, "sem espaço para o checkpoint aqui: aponte HF_CACHE para um caminho com mais espaço, ou use o notebook notebooks/decide/think_gt_api.ipynb"

if os.path.exists(RESUME_FROM) and not os.path.exists(f"{WORK}/results.jsonl"):
    shutil.copy(RESUME_FROM, f"{WORK}/results.jsonl")
    print("retomando de", RESUME_FROM)
os.environ["HF_HOME"] = HF_CACHE

## 5. O pacote é o da amostra
Confere o `pack.jsonl.gz` contra o `gt-sample-manifest.json` (ids e hashes das 400 linhas). Se não bater, pare: a comparação não valeria.

In [ ]:
!python -m decide.think.sample verify --manifest decide/think/gt-sample-manifest.json --pack {PACK}

## 6. Rodar o pensador
Retoma sozinho: se a sessão cair, rode esta célula de novo com os mesmos parâmetros. Para continuar numa sessão nova, use Save Version e adicione a saída dela como Input (`RESUME_FROM`). O orçamento de tokens só pode subir entre execuções; linhas cortadas por ele são refeitas quando ele sobe.

O log não imprime texto de prova: só o id da linha, tokens, segundos, tokens/s e ETA.

In [ ]:
!python -m decide.think.thinker --backend transformers --quant nf4 --batch-size {BATCH_SIZE} --max-tokens {MAX_TOKENS} --pack {PACK} --manifest decide/think/gt-sample-manifest.json --out {WORK}/results.jsonl --time-budget-s {TIME_BUDGET_S} {EXTRA}

## 7. Comparar com os 8 decisores
A última linha é o veredito. Com `LIMIT = "3"` a linha termina em `PARTIAL: ... not run` e o veredito não vale nada: é só para ver que o relatório monta.

In [ ]:
!python -m decide.think.compare --pack {PACK} --thinker {WORK}/results.jsonl --out {WORK}/report

## O que fazer com o resultado
- **PASS**: o pensador entra no plano (ADR 0004). Guarde `report/report.md` e `report/report.json` e abra a issue da próxima fase. O PASS com 4-bit vale; quantizar não costuma melhorar o modelo (UNVERIFIED).
- **FAIL com `can still pass`** na linha: o IC ainda comporta +10; mais linhas ou um orçamento maior podem mudar o resultado.
- **FAIL sem isso**: no 4-bit isso **não prova** que o modelo em bf16 ou FP8 falharia. Confirme pelo `notebooks/decide/think_gt_api.ipynb` (endpoint ou vLLM numa GPU de 96 GB) antes de descartar o pensador.
- `AVISO` no relatório (ruído, orçamento esgotado, execução parcial) vem antes do número.
- Não publique `results.jsonl` nem `pack.jsonl.gz`: têm texto de prova e raciocínio sobre ele.